In [9]:
import pandas as pd
import os, json, librosa
import soundfile as sf
import io
import random

In [ ]:
df = pd.read_parquet(r'D:/NemoASR/rishi70612_audio_data.parquet')
print(df.columns)
print(df.iloc[0])

input_ids→ tokens created for another model,
labels→ likely TTS training data,
input_length→ related to previous processing,
speaker_embedding→ information about the speaker's voice,
path→ old Windows path, probably unusable in Colab

In [ ]:
for i in range(10):
    audio_bytes = df.iloc[i]['audio']['bytes']
    data, sr = sf.read(io.BytesIO(audio_bytes))
    print(i, sr) # Checking whether the audio frequency is 16khz or not

In [ ]:
# Only these two columns matter
df = df[['audio', 'sentence']]

In [ ]:
audio_dir = 'D:/NemoASR/full_run/audio_files'
os.makedirs(audio_dir, exist_ok=True)
manifest_path = 'D:/NemoASR/full_run/full_run_manifest.json'

with open(manifest_path, 'w', encoding='utf-8') as f:
    for i, row in df.iterrows():
        audio_bytes = row['audio']['bytes']
        wav_path = f"{audio_dir}/{i}.wav"
        data, sr = sf.read(io.BytesIO(audio_bytes))

        if sr != 16000:
          data = librosa.resample(data, orig_sr=sr, target_sr=16000)
          sr = 16000

        sf.write(wav_path, data, sr)
        duration = len(data) / sr

        entry = {
            "audio_filepath": wav_path,
            "text": row['sentence'],
            "duration": duration,
            "target_lang": "ne-NP"
        }
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

In [ ]:
with open(manifest_path) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

In [10]:
manifest_path = 'D:/NemoASR/full_run/full_run_manifest.json'

with open(manifest_path, encoding='utf-8') as f:
    lines = f.readlines()

random.seed(42)
random.shuffle(lines)

total = len(lines)

train_end = int(total * 0.8)
val_end = int(total * 0.9)

train_lines = lines[:train_end]
val_lines = lines[train_end:val_end]
test_lines = lines[val_end:]

print(f"Total : {total}")
print(f"Train : {len(train_lines)}")
print(f"Val   : {len(val_lines)}")
print(f"Test  : {len(test_lines)}")

with open('D:/NemoASR/full_run/full_run_train_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(train_lines)

with open('D:/NemoASR/full_run/full_run_val_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(val_lines)

with open('D:/NemoASR/full_run/full_run_test_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(test_lines)

Total : 980
Train : 784
Val   : 98
Test  : 98


In [ ]:
with open('D:/NemoASR/full_run/full_run_train_manifest.json') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

In [2]:
import nemo
import nemo.collections.asr as nemo_asr
print(nemo.__version__)

D:/NemoASR/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


3.1.0+bf014cc8f


In [ ]:
#!git clone -b main https://github.com/NVIDIA/NeMo.git "D:/NemoASR/NeMo"

In [ ]:
# !ls "D:/NemoASR/NeMo/examples/asr"

In [3]:
from nemo.collections.asr.models import ASRModel

model_path = "D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo"

model = ASRModel.restore_from(model_path)


[NeMo I 2026-07-30 10:56:58 mixins:194] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-07-30 10:57:02 modelPT:287] You tried to register an artifact under config key=tokenizer.model_path but an artifact for it has already been registered.
[NeMo W 2026-07-30 10:57:02 modelPT:287] You tried to register an artifact under config key=tokenizer.vocab_path but an artifact for it has already been registered.


[NeMo I 2026-07-30 10:57:02 mixins:194] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-07-30 10:57:07 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    use_lhotse: true
    shard_manifests: true
    batch_duration: 200
    quadratic_duration: 15
    num_buckets: 30
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: true
    tarred_audio_filepaths: null
    shuffle_n: 2048
    slice_length: 100
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    bucket_buffer_size: 10000
    shuffle_buffer_size: 10000
    prompt_field: target_lang
    prompt_dictionary:
      en-US: 0
      en: 0
      en-GB: 1
      enGB: 1
      es-ES: 2
      esES: 2
      es-US: 3
      es: 3
      zh-CN: 4
      zh-ZH: 4
      zh-TW: 5
      hi-IN: 6
      hi: 6
      hi-HI: 6
      ar-AR: 7
     

[NeMo I 2026-07-30 10:57:13 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-07-30 10:57:13 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-07-30 10:57:13 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-07-30 10:57:13 rnnt_bpe_models_prompt:146] Model with prompt feature has been initialized (RNNT-only)
[NeMo I 2026-07-30 10:57:15 save_restore_connector:287] Model EncDecRNNTBPEModelWithPrompt was successfully restored from D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo.


In [4]:
!tar -tf "D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo"

427ad33c6285472cb01c3eb843d2309d_tokenizer.model
c8ead90c911846569df4738620acfc0f_vocab.txt
model_config.yaml
model_weights.ckpt


In [5]:
!mkdir -p D:/NemoASR/extracted_tokenizer
!tar -xf "D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo" \
  -C D:/NemoASR/extracted_tokenizer "427ad33c6285472cb01c3eb843d2309d_tokenizer.model"
!mv "D:/NemoASR/extracted_tokenizer/427ad33c6285472cb01c3eb843d2309d_tokenizer.model" \
    "D:/NemoASR/extracted_tokenizer/tokenizer.model"
!ls -la D:/NemoASR/extracted_tokenizer

!tar -xf "D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo" \
  -C D:/NemoASR/extracted_tokenizer "c8ead90c911846569df4738620acfc0f_vocab.txt"
!mv "D:/NemoASR/extracted_tokenizer/c8ead90c911846569df4738620acfc0f_vocab.txt" \
    "D:/NemoASR/extracted_tokenizer/vocab.txt"
!ls -la D:/NemoASR/extracted_tokenizer


total 700
drwxr-xr-x. 2 root      root          71 Jul 30 10:57 .
drwxr-xr-x. 8 root      root        4096 Jul 23 16:44 ..
-rw-r--r--. 1 361946017 748400513 217588 May 30 05:03 model_config.yaml
-rw-r--r--. 1 361946017 748400513 406554 Feb  4 00:00 tokenizer.model
-rw-r--r--. 1 361946017 748400513  78294 Feb  4 00:05 vocab.txt
total 700
drwxr-xr-x. 2 root      root          71 Jul 30 10:58 .
drwxr-xr-x. 8 root      root        4096 Jul 23 16:44 ..
-rw-r--r--. 1 361946017 748400513 217588 May 30 05:03 model_config.yaml
-rw-r--r--. 1 361946017 748400513 406554 Feb  4 00:00 tokenizer.model
-rw-r--r--. 1 361946017 748400513  78294 Feb  4 00:05 vocab.txt


In [2]:
!tar -xf "D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo" -C D:/NemoASR/extracted_tokenizer "model_config.yaml"

In [6]:
import json

dur = 0
count = 0
with open("D:/NemoASR/full_run/full_run_train_manifest.json") as f:
    for line in f:
        obj = json.loads(line)
        dur += obj["duration"]
        count += 1

print("Files:", count)
print("Total hours:", dur/3600)
print("Average duration:", dur/count)

Files: 882
Total hours: 1.1958057638888886
Average duration: 4.880839852607709


# Finetuning

This section provides fine-tuning workflows for adapting NVIDIA's **Nemotron-3.5-ASR-Streaming-0.6B** model to Nepali (`ne-NP`) speech recognition via cross-lingual prompt warm-initialization (`hi-IN` → `ne-NP`).


## 150-Step Fine-Tuning-Option A: Low-VRAM Run (batch_duration=8s, grad_accum=10)

* **Description**: Memory-optimized warm-start fine-tuning engineered to eliminate CUDA Out-Of-Memory (OOM) crashes on GPUs with limited memory.
* **Objective**: Slices audio batches into smaller 8s chunks (2.5x reduction) and compensates with 10 gradient accumulation steps, maintaining the identical effective batch size ($8\text{s} \times 10 = 80\text{s}$ audio per update) and learning trajectory.
* **Key Parameters**:
  * `max_steps`: **150**
  * `TRAIN_BATCH_DURATION`: **8s**
  * `EVAL_BATCH_DURATION`: **6s** (prevents mid-run validation memory spikes)
  * `accumulate_grad_batches`: **10**
  * `precision`: `bf16-mixed`
  * Memory allocator: `PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"`
* **Use Case**: Essential for GPUs with **$\le 16\text{ GB}$ VRAM** (e.g., Google Colab free T4 GPU, RTX 3080/4080, V100 16GB).
* **Output Artifact**: Saves model checkpoint to `tuned_output/final_after_150steps.nemo`.


In [ ]:
import os
# ---------------------------------------------------------------
# Set BEFORE torch is imported -- the allocator reads this env var
# at first CUDA init. expandable_segments reduces OOM caused by
# fragmentation (common on long runs with varying batch shapes),
# which matters more on a 15GB card with less headroom to absorb it.
# ---------------------------------------------------------------
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import torch
import yaml
import lightning.pytorch as pl
from omegaconf import OmegaConf, open_dict
from nemo.collections.asr.models.rnnt_bpe_models_prompt import EncDecRNNTBPEModelWithPrompt
from nemo.utils.exp_manager import exp_manager
# ---------------------------------------------------------------
# 0. Clean slate -- if this runtime has run any earlier diagnostic
#    cells (restore_from calls, prior `model`/`trainer` objects),
#    those aren't automatically freed. Force it before loading again.
# ---------------------------------------------------------------
for _name in ("model", "trainer"):
    if _name in dir():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------------
# 1. Paths -- adjust if these differ
# ---------------------------------------------------------------
TOKENIZER_DIR = 'D:/NemoASR/extracted_tokenizer'
TOKENIZER_CFG_PATH = f'{TOKENIZER_DIR}/model_config.yaml'
TRAIN_MANIFEST = 'D:/NemoASR/pilot_run/pilot_run_train_manifest.json'
orig_manifest = 'D:/NemoASR/pilot_run/pilot_run_val_manifest.json'
CHECKPOINT_PATH = 'D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo'
EXP_DIR = 'D:/NemoASR/tuned_output'

NE_NP_IDX = 46
HI_IN_IDX = 6
ONE_HOT_OFFSET = 1024

# ---------------------------------------------------------------
# 2. Build model_cfg
# ---------------------------------------------------------------
with open(TOKENIZER_CFG_PATH) as f:
    model_cfg = yaml.safe_load(f)

model_cfg['tokenizer'] = {'dir': TOKENIZER_DIR, 'type': 'bpe'}

# --- MEMORY: batch_duration (seconds of audio per batch) is the main
# VRAM knob for Lhotse's dynamic bucketing dataloader. Cut it down from
# 20s to 8s on train, and add explicit (smaller) values to val/test,
# which previously had no batch_duration set at all and would have
# fallen back to a much larger default -- that's a likely OOM source
# on a 15GB card even if train looked fine.
TRAIN_BATCH_DURATION = 8
EVAL_BATCH_DURATION = 6

model_cfg['train_ds']['manifest_filepath'] = TRAIN_MANIFEST
model_cfg['train_ds']['is_tarred'] = False
model_cfg['train_ds']['shuffle'] = True
model_cfg['train_ds']['batch_duration'] = TRAIN_BATCH_DURATION
model_cfg['train_ds']['num_workers'] = 2
model_cfg['validation_ds']['manifest_filepath'] = orig_manifest
model_cfg['validation_ds']['batch_duration'] = EVAL_BATCH_DURATION
model_cfg['validation_ds']['num_workers'] = 1
model_cfg['test_ds']['manifest_filepath'] = orig_manifest
model_cfg['test_ds']['batch_duration'] = EVAL_BATCH_DURATION
model_cfg['test_ds']['num_workers'] = 1
model_cfg['decoding']['greedy']['use_cuda_graph_decoder'] = False

d_model = model_cfg.get('encoder', {}).get('d_model')
model_cfg['optim'] = {
    'name': 'adamw',
    'lr': 5.0e-5,   # literal peak LR -- WarmupAnnealing has no hidden
                    # d_model scaling, unlike NoamAnnealing.
    'betas': [0.9, 0.98],
    'weight_decay': 1e-3,
    'sched': {
        'name': 'WarmupAnnealing',
        'warmup_steps': 20,
        'min_lr': 1e-7,
    },
}
model_cfg = OmegaConf.create(model_cfg)

# ---------------------------------------------------------------
# 3. Trainer
# --- MEMORY: batch_duration dropped from 20s -> 8s (2.5x smaller),
# so accumulate_grad_batches is raised from 4 -> 10 to keep roughly
# the same *effective* batch size per optimizer step (4*20 ~= 10*8).
# This trades a bit of step-time for a big VRAM reduction, since only
# one micro-batch's activations are ever live at once.
# ---------------------------------------------------------------
trainer_cfg = {
    'devices': 1, 'num_nodes': 1, 'max_epochs': -1,
    'accelerator': 'auto', 'precision': 'bf16-mixed',
    'log_every_n_steps': 10, 'enable_progress_bar': True,
    'num_sanity_val_steps': 0, 'enable_checkpointing': False,
    'logger': False, 'use_distributed_sampler': False,
    'max_steps': 150,
    'limit_train_batches': 50,
    'val_check_interval': 30,
    'gradient_clip_val': 1.0,
    'accumulate_grad_batches': 10,
}

exp_manager_cfg = {
    'exp_dir': EXP_DIR,
    'create_tensorboard_logger': True,
    'create_checkpoint_callback': False,
    'resume_if_exists': False,
    'resume_ignore_no_checkpoint': True,
}

trainer = pl.Trainer(**trainer_cfg)
exp_manager(trainer, OmegaConf.create(exp_manager_cfg))

# ---------------------------------------------------------------
# 4. Restore checkpoint ourselves (no Hydra subprocess -- full control)
# ---------------------------------------------------------------
model = EncDecRNNTBPEModelWithPrompt.restore_from(
    CHECKPOINT_PATH,
    override_config_path=model_cfg,
    trainer=trainer,
)

# --- MEMORY: enable activation checkpointing on the encoder if the
# NeMo/Nemotron encoder class supports it. Guarded with hasattr so this
# is a silent no-op rather than an AttributeError if the API differs
# from what's assumed here -- confirm in your NeMo version whether this
# actually toggles on before relying on it.
if hasattr(model, "encoder") and hasattr(model.encoder, "enable_gradient_checkpointing"):
    model.encoder.enable_gradient_checkpointing()
    print("Gradient checkpointing enabled on encoder.")
else:
    print("Encoder has no enable_gradient_checkpointing() -- skipping "
          "(not an error; batch_duration + accumulate_grad_batches changes "
          "are the primary memory fix here).")

# ---------------------------------------------------------------
# 5. Warm-init: copy hi-IN's prompt_kernel column into ne-NP's slot.
#    Confirmed via diagnostic: ne-NP std=0.0126 (near-untrained) vs
#    hi-IN std=0.268 (well-trained) before this copy.
# ---------------------------------------------------------------
with torch.no_grad():
    w = model.prompt_kernel[0].weight
    print(f"Before warm-init -- ne-NP col std: {w[:, ONE_HOT_OFFSET + NE_NP_IDX].std().item():.6f}, "
          f"hi-IN col std: {w[:, ONE_HOT_OFFSET + HI_IN_IDX].std().item():.6f}")
    w[:, ONE_HOT_OFFSET + NE_NP_IDX] = w[:, ONE_HOT_OFFSET + HI_IN_IDX].clone()
    print(f"After warm-init  -- ne-NP col std: {w[:, ONE_HOT_OFFSET + NE_NP_IDX].std().item():.6f}")

# ---------------------------------------------------------------
# 6. Fix unified_auto_ratio directly on the live model.cfg, then
#    rebuild dataloaders from it -- confirmed necessary, since
#    restore_from() does not reliably honor the YAML-supplied value.
# ---------------------------------------------------------------
with open_dict(model.cfg):
    model.cfg.train_ds.manifest_filepath = TRAIN_MANIFEST
    model.cfg.train_ds.unified_auto_ratio = 0.0
    model.cfg.train_ds.num_workers = 2
    model.cfg.train_ds.batch_duration = TRAIN_BATCH_DURATION
    model.cfg.validation_ds.manifest_filepath = orig_manifest
    model.cfg.validation_ds.unified_auto_ratio = 0.0
    model.cfg.validation_ds.default_prompt_mode = "unified"
    model.cfg.validation_ds.lang_field = "target_lang"
    model.cfg.validation_ds.num_workers = 1
    model.cfg.validation_ds.batch_duration = EVAL_BATCH_DURATION
    model.cfg.test_ds.manifest_filepath = orig_manifest
    model.cfg.test_ds.unified_auto_ratio = 0.0
    model.cfg.test_ds.default_prompt_mode = "unified"
    model.cfg.test_ds.lang_field = "target_lang"
    model.cfg.test_ds.num_workers = 1
    model.cfg.test_ds.batch_duration = EVAL_BATCH_DURATION

model.setup_training_data(model.cfg.train_ds)
model.setup_multiple_validation_data(model.cfg.validation_ds)
model.setup_test_data(model.cfg.test_ds)

print(f"Confirmed val unified_auto_ratio: {model.cfg.validation_ds.get('unified_auto_ratio')}")

# ---------------------------------------------------------------
# 7. Train. With val_check_interval=30, you'll see val predictions
#    periodically through the run -- watch these to see whether
#    output moves from hi-IN-shaped toward actual Nepali over time.
#    Checkpointing is disabled during training (see note above);
#    we save once manually at the end instead, only if the run
#    completes without crashing.
# ---------------------------------------------------------------
trainer.fit(model)

print("\nTraining finished -- saving final checkpoint once, manually.")
final_path = f"{EXP_DIR}/final_after_150steps.nemo"
model.save_to(final_path)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


[NeMo I 2026-07-30 11:05:14 exp_manager:587] ExpManager schema
[NeMo I 2026-07-30 11:05:14 exp_manager:588] {'explicit_log_dir': None, 'exp_dir': None, 'name': None, 'version': None, 'use_datetime_version': True, 'resume_if_exists': False, 'resume_past_end': False, 'resume_ignore_no_checkpoint': False, 'resume_from_checkpoint': None, 'create_tensorboard_logger': True, 'summary_writer_kwargs': None, 'create_wandb_logger': False, 'wandb_logger_kwargs': None, 'create_mlflow_logger': False, 'mlflow_logger_kwargs': {'experiment_name': None, 'run_name': None, 'tracking_uri': None, 'tags': None, 'save_dir': './mlruns', 'prefix': '', 'artifact_location': None, 'run_id': None, 'log_model': False}, 'create_dllogger_logger': False, 'dllogger_logger_kwargs': {'verbose': False, 'stdout': False, 'json_file': './dllogger.json'}, 'create_clearml_logger': False, 'clearml_logger_kwargs': {'project': None, 'task': None, 'connect_pytorch': False, 'model_name': None, 'tags': None, 'log_model': False, 'log_

[NeMo W 2026-07-30 11:05:53 modelPT:287] You tried to register an artifact under config key=tokenizer.model_path but an artifact for it has already been registered.
[NeMo W 2026-07-30 11:05:53 modelPT:287] You tried to register an artifact under config key=tokenizer.vocab_path but an artifact for it has already been registered.


[NeMo I 2026-07-30 11:05:53 mixins:194] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-07-30 11:05:58 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: D:/NemoASR/pilot_run/pilot_run_train_manifest.json
    sample_rate: 16000
    use_lhotse: true
    shard_manifests: true
    batch_duration: 8
    quadratic_duration: 15
    num_buckets: 30
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    slice_length: 100
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    bucket_buffer_size: 10000
    shuffle_buffer_size: 10000
    prompt_field: target_lang
    prompt_dictionary:
      en-US: 0
      en: 0
      en-GB: 1
      enGB: 1
      es-ES: 2
      esES: 2
      es-US: 3
      es: 3
      zh-CN: 4
      zh-ZH: 4
      zh-TW: 5
      hi-IN: 6
  

...

Epoch 0:   4%|▍         | 2/50 [00:06<02:41,  0.30it/s, v_num=5-14, train_step_timing in s=0.417]

[NeMo W 2026-07-30 11:06:19 nemo_logging:364] D:/NemoASR/.venv/lib64/python3.12/site-packages/numba_cuda/numba/cuda/dispatcher.py:720: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
      warn(NumbaPerformanceWarning(msg))
    


Epoch 0:  60%|██████    | 30/50 [00:16<00:11,  1.78it/s, v_num=5-14, train_step_timing in s=0.367][NeMo I 2026-07-30 11:06:30 wer:318] 
    
[NeMo I 2026-07-30 11:06:30 wer:319] WER reference:जनसन एन्ड जनसनले विभिन्न प्रकारका औषधिहरू बजारमा ल्याएको छ
[NeMo I 2026-07-30 11:06:30 wer:320] WER predicted:जॉनसन एंड जॉनसन ले विभिन्न प्रकार का औषधि अरू बाजार में ले आए कुछ था। <hi-IN>
[NeMo I 2026-07-30 11:06:30 wer:318] 
    
[NeMo I 2026-07-30 11:06:30 wer:319] WER reference:नेदरल्यान्ड्सको बेडममा जन्मिएका उनी सानैदेखि दायाँ बिङ्ग्समा खेल्थे
[NeMo I 2026-07-30 11:06:30 wer:320] WER predicted:नीदरलैंड्स को बिडामा जनमी का उन्नीस सानी दिखी दायां बिंग्स माखेल थे। <hi-IN>
[NeMo I 2026-07-30 11:06:30 wer:318] 
    
[NeMo I 2026-07-30 11:06:30 wer:319] WER reference:कोलम्बियाली लेखक र पत्रकार
[NeMo I 2026-07-30 11:06:30 wer:320] WER predicted:कोलंबियाली लेखक्र पत्रकार। <hi-IN>
[NeMo I 2026-07-30 11:06:31 wer:318] 
    
[NeMo I 2026-07-30 11:06:31 wer:319] WER reference:गीतकार एवं कवि विनोद गौचनको श

`Trainer.fit` stopped: `max_steps=150` reached.


Epoch 29: 100%|██████████| 50/50 [00:20<00:00,  2.38it/s, v_num=5-14, train_step_timing in s=0.467]

Training finished -- saving final checkpoint once, manually.


## 150-Step Fine-Tuning-Option B: Standard Baseline Run (`batch_duration = 20s`, `accumulate_grad_batches = 4`)

* **Description**: Primary baseline warm-start fine-tuning setup (equivalent to standalone scripts `train_warmstart.py` and `finetune_150steps_standard.py`).
* **Objective**: Restores the base Nemotron model, copies well-trained Hindi (`hi-IN`) prompt kernel weights into the untrained Nepali (`ne-NP`) slot, and trains for 150 steps using unscaled AdamW `WarmupAnnealing`.
* **Key Parameters**:
  * `max_steps`: **150**
  * `batch_duration`: **20s**
  * `accumulate_grad_batches`: **4** (effective batch audio: $80\text{s}$)
  * `warmup_steps`: **20**
  * `precision`: `bf16-mixed`
* **Use Case**: Recommended for GPUs with **$\ge 24\text{ GB}$ VRAM** (e.g., A100, RTX 3090/4090, A10) or headless execution. Completes training faster due to fewer gradient accumulation passes per update.
* **Output Artifact**: Saves model checkpoint to `tuned_output/final_after_150steps.nemo`.


In [ ]:
import os
# ---------------------------------------------------------------
# Set BEFORE torch is imported -- the allocator reads this env var
# at first CUDA init. expandable_segments reduces OOM caused by
# fragmentation (common on long runs with varying batch shapes),
# which matters more on a 15GB card with less headroom to absorb it.
# ---------------------------------------------------------------
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import gc
import torch
import yaml
import lightning.pytorch as pl
from omegaconf import OmegaConf, open_dict
from nemo.collections.asr.models.rnnt_bpe_models_prompt import EncDecRNNTBPEModelWithPrompt
from nemo.utils.exp_manager import exp_manager
# ---------------------------------------------------------------
# 0. Clean slate -- if this runtime has run any earlier diagnostic
#    cells (restore_from calls, prior `model`/`trainer` objects),
#    those aren't automatically freed. Force it before loading again.
# ---------------------------------------------------------------
for _name in ("model", "trainer"):
    if _name in dir():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------------
# 1. Paths -- adjust if these differ
# ---------------------------------------------------------------
TOKENIZER_DIR = 'D:/NemoASR/extracted_tokenizer'
TOKENIZER_CFG_PATH = f'{TOKENIZER_DIR}/model_config.yaml'
TRAIN_MANIFEST = 'D:/NemoASR/pilot_run/pilot_run_train_manifest.json'
orig_manifest = 'D:/NemoASR/pilot_run/pilot_run_val_manifest.json'
CHECKPOINT_PATH = 'D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo'
EXP_DIR = 'D:/NemoASR/tuned_output'

NE_NP_IDX = 46
HI_IN_IDX = 6
ONE_HOT_OFFSET = 1024

# ---------------------------------------------------------------
# 2. Build model_cfg
# ---------------------------------------------------------------
with open(TOKENIZER_CFG_PATH) as f:
    model_cfg = yaml.safe_load(f)

model_cfg['tokenizer'] = {'dir': TOKENIZER_DIR, 'type': 'bpe'}
model_cfg['train_ds']['manifest_filepath'] = TRAIN_MANIFEST
model_cfg['train_ds']['is_tarred'] = False
model_cfg['train_ds']['shuffle'] = True
model_cfg['train_ds']['batch_duration'] = 20
model_cfg['train_ds']['num_workers'] = 2       # was 8 -- cut subprocess RAM overhead
model_cfg['validation_ds']['manifest_filepath'] = orig_manifest
model_cfg['validation_ds']['num_workers'] = 1  # was 2
model_cfg['test_ds']['manifest_filepath'] = orig_manifest
model_cfg['test_ds']['num_workers'] = 1        # was 8
model_cfg['decoding']['greedy']['use_cuda_graph_decoder'] = False

d_model = model_cfg.get('encoder', {}).get('d_model')
model_cfg['optim'] = {
    'name': 'adamw',
    'lr': 5.0e-5,   # this is now the LITERAL peak LR -- WarmupAnnealing has
                    # no hidden d_model scaling, unlike NoamAnnealing, whose
                    # d_model^-0.5 term silently crushed the last run's
                    # configured 3e-5 down to an effective ~2e-7 (confirmed
                    # from the logged learning_rate scalar: peaked at
                    # 0.00000020, not 0.00003 -- ~150x smaller than intended)
    'betas': [0.9, 0.98],
    'weight_decay': 1e-3,
    'sched': {
        'name': 'WarmupAnnealing',
        'warmup_steps': 20,
        'min_lr': 1e-7,
    },
}
model_cfg = OmegaConf.create(model_cfg)

# ---------------------------------------------------------------
# 3. Trainer -- moderate size for a Colab verification run.
#    Bump these up further once this looks healthy; this is meant
#    to confirm the warm-init lets training actually progress, not
#    to be the final run.
# ---------------------------------------------------------------
trainer_cfg = {
    'devices': 1, 'num_nodes': 1, 'max_epochs': -1,
    'accelerator': 'auto', 'precision': 'bf16-mixed',
    'log_every_n_steps': 10, 'enable_progress_bar': True,
    'num_sanity_val_steps': 0, 'enable_checkpointing': False,
    'logger': False, 'use_distributed_sampler': False,
    'max_steps': 150,
    'limit_train_batches': 50,
    'val_check_interval': 30,   # validate a few times through the run, not just once
    'gradient_clip_val': 1.0,
    'accumulate_grad_batches': 4,
}

exp_manager_cfg = {
    'exp_dir': EXP_DIR,
    'create_tensorboard_logger': True,
    'create_checkpoint_callback': False,   # disabled: packaging the .nemo file
                                            # (~2.5GB) mid-run spikes SYSTEM RAM,
                                            # not GPU VRAM -- this is what crashed
                                            # the last run. Not needed for this
                                            # diagnostic pass; model stays live
                                            # in-process the whole time.
    'resume_if_exists': False,
    'resume_ignore_no_checkpoint': True,
}

trainer = pl.Trainer(**trainer_cfg)
exp_manager(trainer, OmegaConf.create(exp_manager_cfg))

# ---------------------------------------------------------------
# 4. Restore checkpoint ourselves (no Hydra subprocess -- full control)
# ---------------------------------------------------------------
model = EncDecRNNTBPEModelWithPrompt.restore_from(
    CHECKPOINT_PATH,
    override_config_path=model_cfg,
    trainer=trainer,
)

# ---------------------------------------------------------------
# 5. Warm-init: copy hi-IN's prompt_kernel column into ne-NP's slot.
#    Confirmed via diagnostic: ne-NP std=0.0126 (near-untrained) vs
#    hi-IN std=0.268 (well-trained) before this copy.
# ---------------------------------------------------------------
with torch.no_grad():
    w = model.prompt_kernel[0].weight
    print(f"Before warm-init -- ne-NP col std: {w[:, ONE_HOT_OFFSET + NE_NP_IDX].std().item():.6f}, "
          f"hi-IN col std: {w[:, ONE_HOT_OFFSET + HI_IN_IDX].std().item():.6f}")
    w[:, ONE_HOT_OFFSET + NE_NP_IDX] = w[:, ONE_HOT_OFFSET + HI_IN_IDX].clone()
    print(f"After warm-init  -- ne-NP col std: {w[:, ONE_HOT_OFFSET + NE_NP_IDX].std().item():.6f}")

# ---------------------------------------------------------------
# 6. Fix unified_auto_ratio directly on the live model.cfg, then
#    rebuild dataloaders from it -- confirmed necessary, since
#    restore_from() does not reliably honor the YAML-supplied value.
# ---------------------------------------------------------------
with open_dict(model.cfg):
    model.cfg.train_ds.manifest_filepath = TRAIN_MANIFEST
    model.cfg.train_ds.unified_auto_ratio = 0.0
    model.cfg.train_ds.num_workers = 2
    model.cfg.validation_ds.manifest_filepath = orig_manifest
    model.cfg.validation_ds.unified_auto_ratio = 0.0
    model.cfg.validation_ds.default_prompt_mode = "unified"
    model.cfg.validation_ds.lang_field = "target_lang"
    model.cfg.validation_ds.num_workers = 1
    model.cfg.test_ds.manifest_filepath = orig_manifest
    model.cfg.test_ds.unified_auto_ratio = 0.0
    model.cfg.test_ds.default_prompt_mode = "unified"
    model.cfg.test_ds.lang_field = "target_lang"
    model.cfg.test_ds.num_workers = 1

model.setup_training_data(model.cfg.train_ds)
model.setup_multiple_validation_data(model.cfg.validation_ds)
model.setup_test_data(model.cfg.test_ds)

print(f"Confirmed val unified_auto_ratio: {model.cfg.validation_ds.get('unified_auto_ratio')}")

# ---------------------------------------------------------------
# 7. Train. With val_check_interval=30, you'll see val predictions
#    periodically through the run -- watch these to see whether
#    output moves from hi-IN-shaped toward actual Nepali over time.
#    Checkpointing is disabled during training (see note above);
#    we save once manually at the end instead, only if the run
#    completes without crashing.
# ---------------------------------------------------------------
trainer.fit(model)

print("\nTraining finished -- saving final checkpoint once, manually.")
final_path = f"{EXP_DIR}/final_after_150steps.nemo"
model.save_to(final_path)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


[NeMo I 2026-07-30 14:58:51 exp_manager:587] ExpManager schema
[NeMo I 2026-07-30 14:58:51 exp_manager:588] {'explicit_log_dir': None, 'exp_dir': None, 'name': None, 'version': None, 'use_datetime_version': True, 'resume_if_exists': False, 'resume_past_end': False, 'resume_ignore_no_checkpoint': False, 'resume_from_checkpoint': None, 'create_tensorboard_logger': True, 'summary_writer_kwargs': None, 'create_wandb_logger': False, 'wandb_logger_kwargs': None, 'create_mlflow_logger': False, 'mlflow_logger_kwargs': {'experiment_name': None, 'run_name': None, 'tracking_uri': None, 'tags': None, 'save_dir': './mlruns', 'prefix': '', 'artifact_location': None, 'run_id': None, 'log_model': False}, 'create_dllogger_logger': False, 'dllogger_logger_kwargs': {'verbose': False, 'stdout': False, 'json_file': './dllogger.json'}, 'create_clearml_logger': False, 'clearml_logger_kwargs': {'project': None, 'task': None, 'connect_pytorch': False, 'model_name': None, 'tags': None, 'log_model': False, 'log_

[NeMo W 2026-07-30 14:59:44 modelPT:287] You tried to register an artifact under config key=tokenizer.model_path but an artifact for it has already been registered.
[NeMo W 2026-07-30 14:59:44 modelPT:287] You tried to register an artifact under config key=tokenizer.vocab_path but an artifact for it has already been registered.


[NeMo I 2026-07-30 14:59:44 mixins:194] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-07-30 14:59:50 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: D:/NemoASR/pilot_run/pilot_run_train_manifest.json
    sample_rate: 16000
    use_lhotse: true
    shard_manifests: true
    batch_duration: 20
    quadratic_duration: 15
    num_buckets: 30
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    slice_length: 100
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    bucket_buffer_size: 10000
    shuffle_buffer_size: 10000
    prompt_field: target_lang
    prompt_dictionary:
      en-US: 0
      en: 0
      en-GB: 1
      enGB: 1
      es-ES: 2
      esES: 2
      es-US: 3
      es: 3
      zh-CN: 4
      zh-ZH: 4
      zh-TW: 5
      hi-IN: 6
 

...

[NeMo W 2026-07-30 15:00:16 nemo_logging:364] D:/NemoASR/.venv/lib64/python3.12/site-packages/numba_cuda/numba/cuda/dispatcher.py:720: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
      warn(NumbaPerformanceWarning(msg))
    


Epoch 0:  60%|██████    | 30/50 [00:12<00:08,  2.41it/s, v_num=5-14, train_step_timing in s=0.313][NeMo I 2026-07-30 15:00:27 wer:318] 
    
[NeMo I 2026-07-30 15:00:27 wer:319] WER reference:जनसन एन्ड जनसनले विभिन्न प्रकारका औषधिहरू बजारमा ल्याएको छ
[NeMo I 2026-07-30 15:00:27 wer:320] WER predicted:जॉनसन एंड जॉनसन ले विभिन्न प्रकार का औषधि अरू बाजार में ले आए कुछ था। <hi-IN>
[NeMo I 2026-07-30 15:00:27 wer:318] 
    
[NeMo I 2026-07-30 15:00:27 wer:319] WER reference:कोलम्बियाली लेखक र पत्रकार
[NeMo I 2026-07-30 15:00:27 wer:320] WER predicted:कोलंबियाली लेखक्र पत्रकार। <hi-IN>
[NeMo I 2026-07-30 15:00:27 wer:318] 
    
[NeMo I 2026-07-30 15:00:27 wer:319] WER reference:गीतकार एवं कवि विनोद गौचनको शनिवार बिहान पश्चिमाञ्चल क्षेत्रीय अस्पतालमा निधन भएको छ
[NeMo I 2026-07-30 15:00:27 wer:320] WER predicted:गीतकार एवं कॉबी विनोद गौचन को सनिवार बिहाना पश्चिमांचल क्षेत्रीय अस्पताल में निधन भय कुसा। <hi-IN>
[NeMo I 2026-07-30 15:00:28 wer:318] 
    
[NeMo I 2026-07-30 15:00:28 wer:319] WER 

`Trainer.fit` stopped: `max_steps=150` reached.


Epoch 18:  56%|█████▌    | 28/50 [00:12<00:09,  2.26it/s, v_num=5-14, train_step_timing in s=0.546]


: 

## Extended 400-Step Fine-Tuning Run (`final_after_400steps.nemo`)

* **Description**: Extended convergence fine-tuning run scaling the warm-start pipeline from 150 to 400 optimization steps.
* **Objective**: Gives the Conformer-RNNT decoder and prompt embeddings significantly more optimization steps to adapt from Hindi-influenced acoustic priors toward natural, idiomatic Nepali phonetics, improving Word Error Rate (WER) and qualitative accuracy.
* **Key Parameters**:
  * `max_steps`: **400** (~2.7x longer training)
  * `warmup_steps`: **40** (scaled warmup schedule)
  * `val_check_interval`: **40** (10 evenly-spaced validation checkpoints)
  * `batch_duration`: **20s**
  * `accumulate_grad_batches`: **4**
  * `precision`: `bf16`
* **Use Case**: Recommended for long-running Colab or cloud GPU sessions to test if extended training overcomes acoustic drift without overfitting.
* **Output Artifact**: Saves model checkpoint to `D:/NemoASR/tuned_output/final_after_400steps.nemo`.


In [ ]:
# 400 Steps
import gc
import torch
import yaml
import lightning.pytorch as pl
from omegaconf import OmegaConf, open_dict
from nemo.collections.asr.models.rnnt_bpe_models_prompt import EncDecRNNTBPEModelWithPrompt
from nemo.utils.exp_manager import exp_manager

# ---------------------------------------------------------------
# 0. Clean slate -- if this runtime has run any earlier diagnostic
#    cells (restore_from calls, prior `model`/`trainer` objects),
#    those aren't automatically freed. Force it before loading again.
# ---------------------------------------------------------------
for _name in ("model", "trainer"):
    if _name in dir():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------------
TOKENIZER_DIR = 'D:/NemoASR/extracted_tokenizer'
TOKENIZER_CFG_PATH = f'{TOKENIZER_DIR}/model_config.yaml'
TRAIN_MANIFEST = 'D:/NemoASR/pilot_run/pilot_run_train_manifest.json'
orig_manifest = 'D:/NemoASR/pilot_run/pilot_run_val_manifest.json'
CHECKPOINT_PATH = 'D:/NemoASR/nemotron-3.5-asr-streaming-0.6b.nemo'
EXP_DIR = 'D:/NemoASR/tuned_output'

NE_NP_IDX = 46
HI_IN_IDX = 6
ONE_HOT_OFFSET = 1024

# ---------------------------------------------------------------
# 2. Build model_cfg
# ---------------------------------------------------------------
with open(TOKENIZER_CFG_PATH) as f:
    model_cfg = yaml.safe_load(f)

model_cfg['tokenizer'] = {'dir': TOKENIZER_DIR, 'type': 'bpe'}
model_cfg['train_ds']['manifest_filepath'] = TRAIN_MANIFEST
model_cfg['train_ds']['is_tarred'] = False
model_cfg['train_ds']['shuffle'] = True
model_cfg['train_ds']['batch_duration'] = 20
model_cfg['train_ds']['num_workers'] = 2       # was 8 -- cut subprocess RAM overhead
model_cfg['validation_ds']['manifest_filepath'] = orig_manifest
model_cfg['validation_ds']['num_workers'] = 1  # was 2
model_cfg['test_ds']['manifest_filepath'] = orig_manifest
model_cfg['test_ds']['num_workers'] = 1        # was 8
model_cfg['decoding']['greedy']['use_cuda_graph_decoder'] = False

d_model = model_cfg.get('encoder', {}).get('d_model')
model_cfg['optim'] = {
    'name': 'adamw',
    'lr': 5.0e-5,   # this is now the LITERAL peak LR -- WarmupAnnealing has
                    # no hidden d_model scaling, unlike NoamAnnealing, whose
                    # d_model^-0.5 term silently crushed the last run's
                    # configured 3e-5 down to an effective ~2e-7 (confirmed
                    # from the logged learning_rate scalar: peaked at
                    # 0.00000020, not 0.00003 -- ~150x smaller than intended)
    'betas': [0.9, 0.98],
    'weight_decay': 1e-3,
    'sched': {
        'name': 'WarmupAnnealing',
        'warmup_steps': 40,
        'min_lr': 1e-7,
    },
}
model_cfg = OmegaConf.create(model_cfg)

# ---------------------------------------------------------------
# 3. Trainer
# ---------------------------------------------------------------
trainer_cfg = {
    'devices': 1, 'num_nodes': 1, 'max_epochs': -1,
    'accelerator': 'auto', 'precision': 'bf16',
    'log_every_n_steps': 10, 'enable_progress_bar': True,
    'num_sanity_val_steps': 0, 'enable_checkpointing': False,
    'logger': False, 'use_distributed_sampler': False,
    'max_steps': 400,
    'limit_train_batches': 50,
    'val_check_interval': 40,   # fewer, evenly-spaced checks over the longer run
    'gradient_clip_val': 1.0,
    'accumulate_grad_batches': 4,
}

exp_manager_cfg = {
    'exp_dir': EXP_DIR,
    'create_tensorboard_logger': True,
    'create_checkpoint_callback': False,   # disabled: packaging the .nemo file
                                            # (~2.5GB) mid-run spikes SYSTEM RAM,
                                            # not GPU VRAM -- this is what crashed
                                            # the last run. Not needed for this
                                            # diagnostic pass; model stays live
                                            # in-process the whole time.
    'resume_if_exists': False,
    'resume_ignore_no_checkpoint': True,
}

trainer = pl.Trainer(**trainer_cfg)
exp_manager(trainer, OmegaConf.create(exp_manager_cfg))

# ---------------------------------------------------------------
# 4. Restore checkpoint ourselves (no Hydra subprocess -- full control)
# ---------------------------------------------------------------
model = EncDecRNNTBPEModelWithPrompt.restore_from(
    CHECKPOINT_PATH,
    override_config_path=model_cfg,
    trainer=trainer,
)

# ---------------------------------------------------------------
# 5. Warm-init: copy hi-IN's prompt_kernel column into ne-NP's slot.
#    Confirmed via diagnostic: ne-NP std=0.0126 (near-untrained) vs
#    hi-IN std=0.268 (well-trained) before this copy.
# ---------------------------------------------------------------
with torch.no_grad():
    w = model.prompt_kernel[0].weight
    print(f"Before warm-init -- ne-NP col std: {w[:, ONE_HOT_OFFSET + NE_NP_IDX].std().item():.6f}, "
          f"hi-IN col std: {w[:, ONE_HOT_OFFSET + HI_IN_IDX].std().item():.6f}")
    w[:, ONE_HOT_OFFSET + NE_NP_IDX] = w[:, ONE_HOT_OFFSET + HI_IN_IDX].clone()
    print(f"After warm-init  -- ne-NP col std: {w[:, ONE_HOT_OFFSET + NE_NP_IDX].std().item():.6f}")

# ---------------------------------------------------------------
# 6. Fix unified_auto_ratio directly on the live model.cfg, then
#    rebuild dataloaders from it -- confirmed necessary, since
#    restore_from() does not reliably honor the YAML-supplied value.
# ---------------------------------------------------------------
with open_dict(model.cfg):
    model.cfg.train_ds.manifest_filepath = TRAIN_MANIFEST
    model.cfg.train_ds.unified_auto_ratio = 0.0
    model.cfg.train_ds.num_workers = 2
    model.cfg.validation_ds.manifest_filepath = orig_manifest
    model.cfg.validation_ds.unified_auto_ratio = 0.0
    model.cfg.validation_ds.default_prompt_mode = "unified"
    model.cfg.validation_ds.lang_field = "target_lang"
    model.cfg.validation_ds.num_workers = 1
    model.cfg.test_ds.manifest_filepath = orig_manifest
    model.cfg.test_ds.unified_auto_ratio = 0.0
    model.cfg.test_ds.default_prompt_mode = "unified"
    model.cfg.test_ds.lang_field = "target_lang"
    model.cfg.test_ds.num_workers = 1

model.setup_training_data(model.cfg.train_ds)
model.setup_multiple_validation_data(model.cfg.validation_ds)
model.setup_test_data(model.cfg.test_ds)

print(f"Confirmed val unified_auto_ratio: {model.cfg.validation_ds.get('unified_auto_ratio')}")

# ---------------------------------------------------------------
# 7. Train. With val_check_interval=30, you'll see val predictions
#    periodically through the run -- watch these to see whether
#    output moves from hi-IN-shaped toward actual Nepali over time.
#    Checkpointing is disabled during training (see note above);
#    we save once manually at the end instead, only if the run
#    completes without crashing.
# ---------------------------------------------------------------
trainer.fit(model)

print("\nTraining finished -- saving final checkpoint once, manually.")
final_path = f"{EXP_DIR}/final_after_400steps.nemo"
model.save_to(final_path)
print(f"Saved: {final_path}")

# Metrics

This section reads and displays training logs from the saved TensorBoard event files without needing to run the TensorBoard dashboard. It lets you quickly check how the model improved across training steps.

---

### Tracked Metrics & Preferred Values

* **`train_loss`**:
  * **What it is**: Measures the prediction error during training.
  * **Preferred values**: **Lower is better**. It should steadily go down over time. Dropping from ~50 down into the teens indicates stable learning and convergence.

* **`val_wer` (Validation Word Error Rate)**:
  * **What it is**: The percentage of incorrect words on unseen validation audio.
  * **Preferred values**: **Lower is better**. A lower value means better transcription accuracy. A drop from ~1.17 (117% error) down to ~0.50 (50% error) indicates a significant improvement in Nepali speech recognition.

* **`learning_rate`**:
  * **What it is**: Controls the size of weight updates at each step.
  * **Preferred values**: **Smooth warmup followed by gradual decay**. It should ramp up to the configured peak (around $0.00005$) during warmup steps and then decrease toward zero to help the model stabilize.


In [8]:
from tensorboard.backend.event_processing import event_accumulator
import glob
import os

EXP_DIR = 'D:/NemoASR/tuned_output'

# Find the most recent run's tfevents file
run_dirs = sorted(glob.glob(f'{EXP_DIR}/default/*'))
if not run_dirs:
    print(f"No run directories found under {EXP_DIR}/default")
else:
    latest_run = run_dirs[-1]
    print(f"Reading logs from: {latest_run}")

    ea = event_accumulator.EventAccumulator(latest_run)
    ea.Reload()

    print(f"\nAvailable scalars: {ea.Tags()['scalars']}\n")

    if 'train_loss' in ea.Tags()['scalars']:
        print("train_loss trajectory:")
        for e in ea.Scalars('train_loss'):
            print(f"  step {e.step}: {e.value:.4f}")
    else:
        print("No 'train_loss' scalar found -- check the tag names above for the actual name used.")

    if 'val_wer' in ea.Tags()['scalars']:
        print("\nval_wer trajectory:")
        for e in ea.Scalars('val_wer'):
            print(f"  step {e.step}: {e.value:.4f}")

    if 'learning_rate' in ea.Tags()['scalars']:
        print("\nlearning_rate trajectory (sanity check the schedule actually ramped):")
        for e in ea.Scalars('learning_rate'):
            print(f"  step {e.step}: {e.value:.8f}")

Reading logs from: D:/NemoASR/tuned_output/default/2026-07-30_11-05-14

Available scalars: ['hp_metric', 'validation_step_timing in s', 'global_step', 'val_wer', 'epoch', 'train_loss', 'learning_rate', 'train_backward_timing in s', 'train_step_timing in s', 'training_batch_wer']

train_loss trajectory:
  step 9: 50.7924
  step 19: 49.6125
  step 29: 23.3868
  step 39: 94.1287
  step 49: 49.8734
  step 59: 13.8890
  step 69: 44.4278
  step 79: 84.1742
  step 89: 8.4080
  step 99: 11.3988
  step 109: 42.8923
  step 119: 25.3503
  step 129: 33.8034
  step 139: 27.9094
  step 149: 24.4809

val_wer trajectory:
  step 2: 1.1719
  step 7: 1.1719
  step 12: 1.1406
  step 17: 0.8594
  step 22: 0.7500
  step 27: 0.7188
  step 32: 0.6250
  step 37: 0.6094
  step 42: 0.5938
  step 47: 0.5938
  step 52: 0.5625
  step 57: 0.5938
  step 62: 0.6094
  step 67: 0.5781
  step 72: 0.5469
  step 77: 0.5156
  step 82: 0.5000
  step 87: 0.5312
  step 92: 0.5312
  step 97: 0.5312
  step 102: 0.5469
  step 107

# Testing of the Tuned Model
We test in two ways: **Quantitative** (measuring the error rate number) and **Qualitative** (actually reading the Nepali output).


## Quantitative Test (WER)
Runs `trainer.test()` on `full_run_test_manifest.json` to calculate **Word Error Rate (WER)**.
* **Goal**: An objective number to compare runs.
* **Preferred**: **Lower is better** (e.g. `0.48` = 48% error).
* *Note*: Scores alone don't prove language accuracy since Nepali and Hindi share Devanagari script.


In [14]:
import gc
import torch
import lightning.pytorch as pl
from omegaconf import open_dict
from nemo.collections.asr.models.rnnt_bpe_models_prompt import EncDecRNNTBPEModelWithPrompt

# ---------------------------------------------------------------
# 0. Paths -- point at the checkpoint the training script saved
# ---------------------------------------------------------------
FINAL_CHECKPOINT = 'D:/NemoASR/tuned_output/final_after_150steps.nemo'
TEST_MANIFEST = 'D:/NemoASR/full_run/full_run_test_manifest.json'

NE_NP_IDX = 46
HI_IN_IDX = 6
ONE_HOT_OFFSET = 1024

# ---------------------------------------------------------------
# 1. Load fresh -- a clean load (not the in-memory training `model`
#    object) confirms the .nemo file was actually saved correctly
#    and is loadable standalone, which matters if you plan to deploy it.
# ---------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

trainer = pl.Trainer(devices=1, accelerator='auto', precision='bf16-mixed', logger=False)
model = EncDecRNNTBPEModelWithPrompt.restore_from(FINAL_CHECKPOINT, trainer=trainer)
model.eval()

# ---------------------------------------------------------------
# 2. Quick check: did the warm-init survive the save/reload?
#    Compare against your earlier diagnostic numbers
#    (pre-warm-init: ne-NP std=0.0126, hi-IN std=0.268).
# ---------------------------------------------------------------
with torch.no_grad():
    w = model.prompt_kernel[0].weight
    print(f"Reloaded model -- ne-NP col std: {w[:, ONE_HOT_OFFSET + NE_NP_IDX].std().item():.6f}, "
          f"hi-IN col std: {w[:, ONE_HOT_OFFSET + HI_IN_IDX].std().item():.6f}")

# ---------------------------------------------------------------
# 3. Quantitative WER on the test manifest
# ---------------------------------------------------------------
with open_dict(model.cfg):
    model.cfg.test_ds.manifest_filepath = TEST_MANIFEST
    model.cfg.test_ds.unified_auto_ratio = 0.0
    model.cfg.test_ds.default_prompt_mode = "unified"
    model.cfg.test_ds.lang_field = "target_lang"
    model.cfg.test_ds.num_workers = 1
    model.cfg.test_ds.batch_duration = 6

model.setup_test_data(model.cfg.test_ds)
test_results = trainer.test(model)
print("\nTest results:", test_results)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


[NeMo I 2026-07-30 12:31:52 mixins:194] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-07-30 12:31:56 modelPT:287] You tried to register an artifact under config key=tokenizer.model_path but an artifact for it has already been registered.
[NeMo W 2026-07-30 12:31:56 modelPT:287] You tried to register an artifact under config key=tokenizer.vocab_path but an artifact for it has already been registered.


[NeMo I 2026-07-30 12:31:56 mixins:194] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-07-30 12:32:02 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: D:/NemoASR/pilot_run/pilot_run_train_manifest.json
    sample_rate: 16000
    use_lhotse: true
    shard_manifests: true
    batch_duration: 8
    quadratic_duration: 15
    num_buckets: 30
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    slice_length: 100
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    bucket_buffer_size: 10000
    shuffle_buffer_size: 10000
    prompt_field: target_lang
    prompt_dictionary:
      en-US: 0
      en: 0
      en-GB: 1
      enGB: 1
      es-ES: 2
      esES: 2
      es-US: 3
      es: 3
      zh-CN: 4
      zh-ZH: 4
      zh-TW: 5
      hi-IN: 6
  

...

[NeMo I 2026-07-30 12:32:16 dataloader:659] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=6.0 max_batch_size=16)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 0: |          | 0/? [00:00<?, ?it/s][NeMo I 2026-07-30 12:32:18 wer:318] 
    
[NeMo I 2026-07-30 12:32:18 wer:319] WER reference:आकाश गिरी नेपाली चलचित्र क्षेत्रको एक चर्चित अभिनेता हुन्
[NeMo I 2026-07-30 12:32:18 wer:320] WER predicted:आकाशगिरि नेपाली चलचित्र क्षेत्रको एक चर्चित अभिनेता हुन
Testing DataLoader 0: |          | 1/? [00:00<00:00,  4.43it/s][NeMo I 2026-07-30 12:32:18 wer:318] 
    
[NeMo I 2026-07-30 12:32:18 wer:319] WER reference:दुई हजार सातमा भएको मृत्यु
[NeMo I 2026-07-30 12:32:18 wer:320] WER predicted:दुई हजार सातमा भएको मृत्यु
Testing DataLoader 0: |          | 2/? [00:00<00:00,  4.66it/s][NeMo I 2026-07-30 12:32:19 wer:318] 
    
[NeMo I 2026-07-30 12:32:19 wer:319] WER reference:टर्की महिला मध्यम दुरी धावक
[NeMo I 2026-07-30 12:32:19 wer:320] WER predicted:टर्की महिला मध्यम दुरी धावक
Testing DataLoader 0: |          | 3/? [00:00<00:00,  4.50it/s][NeMo I 2026-07-30 12:32:19 wer:318] 
    
[NeMo I 2026-07-30 12:32:19 wer:319] WER reference:खप्

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        global_step        │            0.0            │
│         test_wer          │    0.47815820574760437    │
└───────────────────────────┴───────────────────────────┘


Test results: [{'global_step': 0.0, 'test_wer': 0.47815820574760437}]


## Qualitative Test (Transcription Check)
Transcribes sample audio clips (`namaste.wav`, `kathmandu.wav`, etc.) so a human can read the predictions.
* **Goal**: Ensure the model produces natural **Nepali**, not Hindi bleed (e.g. *"मौसम निकै राम्रो छ"* vs *"मौसम बहुत अच्छा है"*) or hallucinations during silence.
* **Preferred**: Accurate, natural Nepali text.


In [ ]:
audio, sr = librosa.load("D:/NemoASR/Recording.m4a", sr=16000, mono=True)
sf.write("Typing.wav", audio, sr)

[NeMo W 2026-07-30 14:48:45 nemo_logging:364] /tmp/ipykernel_2787187/1765189956.py:1: UserWarning: PySoundFile failed. Trying audioread instead.
      audio, sr = librosa.load("D:/NemoASR/Relation.m4a", sr=16000, mono=True)
    


In [25]:
import json
import tempfile
import os
import torch

# ---------------------------------------------------------------
# Fix for: ValueError: Unknown prompt key: 'None'
#
# Cause: model.transcribe(raw_wav_paths) builds Lhotse cuts with no
# supervision.language set (no manifest = no target_lang to read),
# so it defaults to the string 'None', which isn't in prompt_dict.
# The custom prompt-index dataset class only gets a valid language
# when cuts are built FROM a manifest with lang_field configured --
# exactly what your train/val/test_ds pipeline already does correctly.
#
# Fix: build a manifest for the holdout files with target_lang set,
# and route through setup_test_data() / the test dataloader instead
# of the raw-path transcribe() API.
# ---------------------------------------------------------------

sample_audio_files = [
    'D:/NemoASR/TestingAudioFiles/kathmandu.wav',
    'D:/NemoASR/TestingAudioFiles/namaste.wav',
    'D:/NemoASR/TestingAudioFiles/Nothing.wav',
    'D:/NemoASR/TestingAudioFiles/Relation.wav'
]

TARGET_LANG = "ne-NP"

# NeMo manifests require a duration field; get it via soundfile/torchaudio
# rather than assuming -- wrong duration can silently affect Lhotse bucketing.
import soundfile as sf

manifest_entries = []
for path in sample_audio_files:
    info = sf.info(path)
    duration = info.frames / info.samplerate
    print(f"  {path}: {duration:.2f}s")
    manifest_entries.append({
        "audio_filepath": path,
        "duration": duration,
        "text": "",          # unknown ground truth -- fine for transcription-only use
        "target_lang": TARGET_LANG,
    })

tmp_manifest_path = tempfile.mktemp(suffix=".json")
with open(tmp_manifest_path, "w") as f:
    for entry in manifest_entries:
        f.write(json.dumps(entry) + "\n")

print(f"Wrote holdout manifest with {len(manifest_entries)} entries "
      f"(target_lang='{TARGET_LANG}') to {tmp_manifest_path}")
if len(manifest_entries) != len(sample_audio_files):
    print("WARNING: entry count doesn't match sample_audio_files length -- "
          "check for duplicate/skipped paths.")

# ---------------------------------------------------------------
# Route through the same config path that already worked for
# trainer.test() -- this is what actually gets lang_field honored.
# ---------------------------------------------------------------
from omegaconf import open_dict

with open_dict(model.cfg):
    model.cfg.test_ds.manifest_filepath = tmp_manifest_path
    model.cfg.test_ds.unified_auto_ratio = 0.0
    model.cfg.test_ds.default_prompt_mode = "unified"
    model.cfg.test_ds.lang_field = "target_lang"
    model.cfg.test_ds.num_workers = 1
    model.cfg.test_ds.batch_duration = 21  # inference-only: generous ceiling so no
                                            # individual file gets excluded for
                                            # exceeding max_batch_duration (this was
                                            # likely why only 503.wav, the shortest
                                            # file, made it through at batch_duration=6)
    model.cfg.test_ds.shuffle = False  # keep output order matching input order

model.setup_test_data(model.cfg.test_ds)

# ---------------------------------------------------------------
# Run inference over the test dataloader manually so we get text
# output aligned with file order, rather than just an aggregate WER.
# ---------------------------------------------------------------
model.eval()

# ---------------------------------------------------------------
# The manifest fix above resolved the 'Unknown prompt key' error --
# the dataloader now produces a valid prompt_indices tensor for
# ne-NP. The NEW error (ValueError: prompt_indices must be provided
# when concat mode is enabled) is just that we weren't passing it
# through to forward(). Inspect the batch structure once before
# trusting a guessed tuple position a second time.
# ---------------------------------------------------------------
_debug_batch = next(iter(model._test_dl))
print("Batch has", len(_debug_batch), "elements:")
for i, item in enumerate(_debug_batch):
    if torch.is_tensor(item):
        print(f"  [{i}] tensor shape={tuple(item.shape)} dtype={item.dtype}")
    else:
        print(f"  [{i}] type={type(item)}")

# Based on __getitem__ in audio_to_text_lhotse_prompt_index.py building
# (tokens, token_lens, prompt_indices) after (signal, signal_len), the
# collated order is very likely:
#   [0] input_signal, [1] input_signal_length,
#   [2] tokens, [3] token_lengths, [4] prompt_indices
# CONFIRM this against the printed shapes above -- prompt_indices should
# be a 1D long tensor of length batch_size, every value == NE_NP_IDX (46)
# since every held-out file was tagged target_lang='ne-NP' above.
PROMPT_INDICES_POSITION = 4

decoded_texts = []
with torch.no_grad():
    for batch in model._test_dl:
        batch = [x.to(model.device) if torch.is_tensor(x) else x for x in batch]
        input_signal, input_signal_length = batch[0], batch[1]
        prompt_indices = batch[PROMPT_INDICES_POSITION]

        encoded, encoded_len = model.forward(
            input_signal=input_signal,
            input_signal_length=input_signal_length,
            prompt_indices=prompt_indices,
        )
        best_hyp = model.decoding.rnnt_decoder_predictions_tensor(
            encoded, encoded_len, return_hypotheses=False,
        )
        decoded_texts.extend(best_hyp)

if decoded_texts:
    for path, text in zip(sample_audio_files, decoded_texts):
        print(f"\n{path}\n  -> {text}")
else:
    print("\nNo output -- check PROMPT_INDICES_POSITION against the printed batch "
          "shapes above; a 1D long tensor of length batch_size, all values 46, is "
          "prompt_indices. If it's at a different position, update the constant "
          "and rerun. If decoding still fails, trainer.test(model) with this "
          "manifest (already wired above) gives an aggregate WER on these files "
          "without needing the manual forward() call at all.")

os.remove(tmp_manifest_path)

  D:/NemoASR/TestingAudioFiles/kathmandu.wav: 4.03s
  D:/NemoASR/TestingAudioFiles/namaste.wav: 2.88s
  D:/NemoASR/TestingAudioFiles/Nothing.wav: 3.43s
  D:/NemoASR/TestingAudioFiles/Relation.wav: 9.94s
Wrote holdout manifest with 4 entries (target_lang='ne-NP') to /tmp/tmpdgnm2bps.json
[NeMo I 2026-07-30 14:56:21 audio_to_text_lhotse_prompt_index:86] LhotseSpeechToTextBpeDatasetWithPromptIndex: default_prompt_mode=unified, unified_auto_ratio=0.0
[NeMo I 2026-07-30 14:56:21 rnnt_bpe_models_prompt:181] Setting up Lhotse dataset with prompt index support (RNNT-only model creates prompt tensors)
[NeMo I 2026-07-30 14:56:21 dataloader:342] We will be using a Lhotse DataLoader.


[NeMo W 2026-07-30 14:56:21 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: prompt_field,default_prompt_mode,num_prompts,subsampling_factor,training_mode,prompt_dictionary,use_start_end_token,unified_auto_ratio
[NeMo W 2026-07-30 14:56:21 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[NeMo I 2026-07-30 14:56:21 dataloader:659] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=21.0 max_batch_size=16)
Batch has 5 elements:
  [0] tensor shape=(4, 159062) dtype=torch.float32
  [1] tensor shape=(4,) dtype=torch.int32
  [2] tensor shape=(4, 0) dtype=torch.float32
  [3] tensor shape=(4,) dtype=torch.int64
  [4] tensor shape=(4,) dtype=torch.int64

D:/NemoASR/TestingAudioFiles/kathmandu.wav
  -> Hypothesis(score=-11.022674560546875, y_sequence=tensor([3272, 3248, 3355, 3281, 3248, 3295, 3250, 3259, 3250, 3261, 3250, 3268,
        3304, 3250, 3261, 3358, 3276]), text='बाँचा कानपन्नो जान्छू', dec_out=None, dec_state=None, timestamp=[], alignments=None, frame_confidence=None, token_confidence=None, word_confidence=None, length=0, y=None, lm_state=None, lm_scores=None, ngram_lm_state=None, tokens=None, last_token=None, token_duration=None, last_frame=None, biasing_cfg=None, non_blank_step_confidence_precomputed=None, xatt_scores=None)

D:/NemoASR/